<a href="https://colab.research.google.com/github/abedadba/CNN_Classification_model_dogs_cats/blob/main/CNN_model_dogs_and_cats_classifcation_github_upload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Problem statement :
In this Section we are implementing Convolution Neural Network(CNN) Classifier for Classifying dog and cat images. The Total number of images available for training is 25,000 and final testing is done on 1000 images.

Note:This problem statement and dataset is taken from this Kaggle competition.
Dependencies
Jupyter notebook
Tensorflow 1.10
Python 3.6
Matplotlib
Seaborn
Scikit-Learn
Pandas
Numpy
Install dependencies using conda

Test Train Split
Image training set contain 12500 images for each category. I split those into train and test. Split each class images into 12,000 for train and 500 for test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import cv2
import os
import shutil
from tqdm.auto import tqdm

In [ ]:
!kaggle competitions download -c dogs-vs-cats

In [ ]:
! unzip -q dogs-vs-cats.zip
! unzip -q train.zip

In [ ]:
# Create master and cateory folders
os.mkdir("images")
os.mkdir("images/dog")
os.mkdir("images/cat")
os.mkdir("test")

In [ ]:
# transfer images from Train folder to images/cat & images/dog
source = "train/"
dest_cat = "images/cat/"
dest_dog = "images/dog/"

for imageName in tqdm(os.listdir(source)):
  if imageName.startswith("dog"):
    shutil.copy(source + imageName, dest_dog)
  elif imageName.startswith("cat"):
    shutil.copy(source + imageName, dest_cat)

In [ ]:
len(os.listdir(dest_dog)), len(os.listdir(dest_cat))

In [ ]:
test_dog = np.random.choice(os.listdir(dest_dog), 500, replace=False)
test_cat = np.random.choice(os.listdir(dest_cat), 500, replace=False)

for i in test_dog:
  shutil.move(dest_dog + i, "test/")
for i in test_cat:
  shutil.move(dest_cat + i, "test/")


In [ ]:
len(os.listdir(dest_dog)), len(os.listdir(dest_cat))

In [ ]:
len(os.listdir("test/"))

In [ ]:
idg = tf.keras.preprocessing.image.ImageDataGenerator(horizontal_flip=True,
                                                      rotation_range=30, rescale=1/255.0, validation_split=0.1)

In [ ]:
train_idg = idg.flow_from_directory("images/", target_size=(150, 150), batch_size=64, subset="training",)

In [ ]:
val_idg = idg.flow_from_directory("images", target_size=(150, 150), batch_size=64, subset="validation")

In [ ]:
from tensorflow.keras import layers,models

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input((150,150,3), name="Input"))

In [ ]:
model = models.Sequential([
    layers.Input((150,150,3), name="Input"),
    layers.Conv2D(filters = 16, kernel_size=(3,3), padding="valid", strides=(1,1), activation="relu", name="Conv1",),
    layers.MaxPooling2D(pool_size=(2,2), strides=(2,2), padding="valid", name='Pool1'),
    layers.Conv2D(filters = 32, kernel_size=(3,3), padding="valid", strides=(1,1), activation="relu", name='Conv2'),
    layers.MaxPooling2D(pool_size=(2,2), strides=(2,2), padding="valid", name='Pool2'),
    layers.Conv2D(filters = 32, kernel_size=(3,3), padding="valid", strides=(1,1), activation="relu", name='Conv3'),
    layers.MaxPooling2D(pool_size=(2,2), strides=(2,2), padding="valid", name='Pool3'),
    layers.Flatten(name='Flat'),
    layers.Dense(64, activation="relu", name="Dense1"),
    layers.Dense(2, activation="softmax", name="Output")
])

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.optimizers import SGD

In [ ]:
model.compile(optimizer = "SGD", loss = 'categorical_crossentropy', metrics =['accuracy'])

In [ ]:
#model.fit(train_idg, epochs=10, batch_size = 64, validation_data=val_idg)
model.fit(train_idg, epochs=10, validation_data=val_idg)

There are no signs of overfitting seen as training and validation loss steadily decreased.

**Model Evaluation:**

In [ ]:
test_loss, test_acc = model.evaluate(val_idg)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 1. Reset generator and get predictions
val_idg.reset()
predictions = model.predict(val_idg)

# 2. Convert probabilities to class indices (0 or 1)
# For categorical (softmax), use argmax. For binary (sigmoid), use > 0.5.
predicted_classes = np.argmax(predictions, axis=1)

# 3. Get true labels from the generator
true_classes = val_idg.classes
class_labels = list(val_idg.class_indices.keys()) # ['cat', 'dog']

# 4. Detailed Report
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

In [ ]:
def predict_single_image(img_path, model):
    # 1. Check if the file exists
    if not os.path.exists(img_path):
        print(f"Error: The file '{img_path}' does not exist.")
        return

    # 2. Load the image
    img = cv2.imread(img_path)

    # 3. Check if OpenCV successfully read it
    if img is None:
        print(f"Error: OpenCV could not read '{img_path}'. It might be a broken file.")
        return

    # Now it is safe to proceed
    img_resized = cv2.resize(img, (150, 150))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    img_tensor = np.expand_dims(img_rgb, axis=0) / 255.0
    pred = model.predict(img_tensor)
    idx = np.argmax(pred)

    label = "Dog" if idx == 1 else "Cat"
    plt.imshow(img_rgb)
    plt.title(f"Prediction: {label}")
    plt.show()

**Prediction Test:**

In [ ]:
predict_single_image("test/cat.10030.jpg", model)

In [ ]:
predict_single_image("test/dog.11442.jpg", model)

In [ ]:
predict_single_image("test/cat.899.jpg", model)

In [ ]:
import os

img_path = "test/cat.899.jpg"

# 1. Check if the file exists on disk first
if not os.path.exists(img_path):
    print(f"Error: File not found at {img_path}")
else:
    test_image = cv2.imread(img_path)

    # 2. Check if OpenCV actually loaded it (returns None if corrupted or invalid)
    if test_image is None:
        print("Error: OpenCV could not read the image. Check file integrity.")
    else:
        # Now it is safe to resize
        test_image = cv2.resize(test_image, (150, 150))
        test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
        plt.imshow(test_image)
        test_image = np.expand_dims(test_image, axis=0)
        test_image = test_image / 255.0
        test_image.shape

In [ ]:
history = model.fit(
    train_idg,
    epochs=10,
    validation_data=val_idg  # This triggers validation at each epoch
)

In [ ]:
plt.plot(history.history['accuracy'], label = 'Training Accuracy')
plt.plot(history.history['val_accuracy'], label = 'Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.show()

In [ ]:
def predict_animal(img_path, model):
    # 1. Validation check
    if not os.path.exists(img_path):
        return f"Error: File '{img_path}' not found."

    # 2. Read and Preprocess
    img = cv2.imread(img_path)
    if img is None:
        return "Error: Could not read image."

    # Resize to match model input (150x150)
    img_resized = cv2.resize(img, (150, 150))
    # Convert BGR (OpenCV) to RGB (Model training format)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

    # Scale pixels (0-1) and add Batch Dimension (1, 150, 150, 3)
    img_tensor = np.expand_dims(img_rgb, axis=0) / 255.0

    # 3. Predict
    prediction = model.predict(img_tensor)
    class_idx = np.argmax(prediction[0]) # Get index of highest probability
    confidence = prediction[0][class_idx] * 100

    # 4. Map index to label (0: Cat, 1: Dog)
    label = "Dog" if class_idx == 1 else "Cat"

    # 5. Display Result
    plt.imshow(img_rgb)
    plt.title(f"Prediction: {label} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()

    return label

**Prediction of Online Images**:
The model has predicted successfully the images of cat and dog with confidence above 86%, proves the model is a successful model.

In [ ]:
# Usage
result = predict_animal("/content/images/Prediction_Images/cat_image_test.jpg", model)

In [ ]:
result = predict_animal("/content/images/Prediction_Images/dog_image_test.jpg", model)

In [ ]:
def classify_folder(folder_path, model):
    valid_extensions = ('.jpg', '.jpeg', '.png')
    results = {"Dog": [], "Cat": []}

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(valid_extensions):
            full_path = os.path.join(folder_path, filename)

            # Simplified preprocessing
            img = cv2.imread(full_path)
            img = cv2.resize(img, (150, 150))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_tensor = np.expand_dims(img, axis=0) / 255.0

            pred = model.predict(img_tensor, verbose=0)
            label = "Dog" if np.argmax(pred) == 1 else "Cat"
            results[label].append(filename)

    print(f"Summary: Found {len(results['Dog'])} Dogs and {len(results['Cat'])} Cats.")
    return results

In [ ]:
# 1. Path to your folder of new images
test_folder_path = "/content/images/Prediction_Images"

# 2. Call the function (ensure your 'model' is already loaded/trained)
# This assumes the classify_folder function from the previous response is defined
predictions_summary = classify_folder(test_folder_path, model)

# 3. View the results
print("\n--- Prediction Results ---")
print(f"Total Dogs found: {len(predictions_summary['Dog'])}")
print(f"Total Cats found: {len(predictions_summary['Cat'])}")

# Optional: Print names of images predicted as Dogs
print("\nImages classified as Dogs:")
for dog_img in predictions_summary['Dog'][:5]: # Showing first 5
    print(f"- {dog_img}")